# used liberaries

In [ ]:
# USED LIBRARIES
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import nltk
import gc
import os

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Machine Learning
from sklearn.feature_extraction.text import TfidfVectorizer, HashingVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, roc_curve, auc, confusion_matrix
import xgboost as xgb
from wordcloud import WordCloud
from sklearn.metrics import precision_recall_curve, average_precision_score
# Deep Learning
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Dropout, Embedding, LSTM, Conv1D, MaxPooling1D, GlobalMaxPooling1D, Input, Bidirectional, SpatialDropout1D, BatchNormalization
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

import torch
from transformers import DistilBertTokenizer, DistilBertModel 
from transformers import AutoTokenizer, AutoModel
from transformers import TFDistilBertForSequenceClassification, AdamWeightDecay


In [ ]:
# Download NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

# load and describe dataset

In [ ]:
df = pd.read_csv(r'/kaggle/input/fake-news-classification/WELFake_Dataset.csv')

In [ ]:
print("Dataset shape:", df.shape)

In [ ]:
print("First 20 rows:")
print(df.head(20))

In [ ]:
# Check for missing values
print("Missing values:")
print(df.isnull().sum())

In [ ]:
# Handle missing values
df = df.dropna(subset=['text', 'title'])

In [ ]:
# Combine title and text
df['full_text'] = df['title'] + ' ' + df['text']
df = df[['full_text', 'label']]

In [ ]:
df.duplicated().sum()

In [ ]:
df = df.drop_duplicates()
print("Dataset after cleaning:", df.shape)

In [ ]:
df

# Preprocessing

In [ ]:
# TEXT CLEANING FUNCTIONS
def remove_urls(text):
    return re.sub(r'http\S+|www\.\S+', '', text)

def remove_html_tags(text):
    return re.sub(r'<.*?>', '', text)

def convert_to_lowercase(text):
    return text.lower()

def remove_punctuation(text):
    #keep apostrophes for contractions
    return text.translate(str.maketrans('', '', string.punctuation.replace("'", "")))

def remove_numbers(text):
    return re.sub(r'\b\d+\b', '', text)

def remove_extra_whitespace(text):
    return ' '.join(text.split())

def remove_special_characters (text):
    return re.sub(r'[^a-zA-Z\s]', ' ', text)

In [ ]:
def tokenize_text(text):
    try:
        return word_tokenize(text)
    except LookupError:
       
        return text.split()
def remove_stopwords(tokens):
    stop_words = set(stopwords.words('english'))
    return [word for word in tokens if word not in stop_words]
def lemmatize_tokens(tokens):
    lemmatizer = WordNetLemmatizer()
    return [lemmatizer.lemmatize(word) for word in tokens]

In [ ]:
def clean_text(text):
    # Text cleaning
    text = remove_urls(text)
    text = remove_html_tags(text)
    text = convert_to_lowercase(text)
    text = remove_punctuation(text)
    text = remove_numbers(text)
    text = remove_special_characters(text)
    text = remove_extra_whitespace(text)
    
    tokens = tokenize_text(text)
    
    tokens = remove_stopwords(tokens)
    
    tokens = lemmatize_tokens(tokens)
    
    # Join tokens back into string
    return ' '.join(tokens)

In [ ]:
#APPLY
df['clean_text'] = df['full_text'].apply(clean_text)

# Remove rows where processed text is empty
df = df[df['clean_text'].str.strip() != '']


print("\nOriginal text:")
print(df['full_text'].iloc[0][:200])
print("\nProcessed text:")
print(df['clean_text'].iloc[0][:200])

In [ ]:
df.head()

# visualization

In [ ]:
# VISUALIZATION
plt.figure(figsize=(8, 5))
sns.countplot(x='label', data=df)
plt.title('Distribution of News (0=Fake, 1=Real)')
plt.xlabel('Label')
plt.ylabel('Count')
plt.show()

# Split 

In [ ]:
X = df['clean_text']
y = df['label']

# splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Data splits:')
print(f'Train: {len(X_train)} | Test: {len(X_test)}')

# Tf idf

In [ ]:

tfidf = TfidfVectorizer(
    max_features=15000,        
    ngram_range=(1, 3),        # unigrams, bigrams, and trigrams
    min_df=2,                  # Ignore very rare words
    max_df=0.9,                # Ignore very common words
    sublinear_tf=True,    
    use_idf=True,              
    norm='l2'                  # L2 normalization
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)


# word embadding

In [ ]:
max_words = 20000
max_len = 300      

tokenizer = Tokenizer(num_words=max_words, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

In [ ]:
# Convert text to sequences
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len , padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len , padding='post', truncating='post')
embedding_dim = 100

In [ ]:
print(f"Vocabulary size: {len(tokenizer.word_index)}")
print(f"Using top {max_words} words")
print(f"Sequence shape: {X_train_pad.shape}")

# contextual embeddings 

In [ ]:
model_name = 'distilbert-base-uncased'
tokenizer = DistilBertTokenizer.from_pretrained(model_name)

def tokenize_data(texts, max_length=256):
    return tokenizer(
        texts.tolist(),
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_tensors='tf'
    )

In [ ]:
X_train_encoded = tokenize_data(X_train, max_length=256)
X_test_encoded = tokenize_data(X_test, max_length=256)

# model ( logistic reg )

In [ ]:
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_tfidf, y_train)

# Predict
y_pred_lr = lr_model.predict(X_test_tfidf)
# Predict on training data
y_pred_train = lr_model.predict(X_train_tfidf)

print(f"Train Accuracy: {accuracy_score(y_train, y_pred_train):.4f}")
print(f"Test Accuracy:  {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"\nLogistic Regression Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_lr):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_lr):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_lr):.4f}")
print("\n" + classification_report(y_test, y_pred_lr, target_names=['Real', 'Fake'], digits=4))


In [ ]:
lr_accuracy = accuracy_score(y_test, y_pred_lr)
lr_precision = precision_score(y_test, y_pred_lr)
lr_recall = recall_score(y_test, y_pred_lr)
lr_f1 = f1_score(y_test, y_pred_lr)

# Confusion Matrix
cm_lr = confusion_matrix(y_test, y_pred_lr)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Purples', 
            xticklabels=['Real', 'Fake'], 
            yticklabels=['Real', 'Fake'],
            cbar_kws={'label': 'Count'})
plt.title(f'Logistic Regression - Confusion Matrix\nAccuracy: {lr_accuracy:.4f}', 
          fontsize=14, fontweight='bold')
plt.ylabel('Actual', fontsize=12)
plt.xlabel('Predicted', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
#Bar Chart
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
values_lr = [lr_accuracy, lr_precision, lr_recall, lr_f1]

plt.figure(figsize=(10, 6))
bars = plt.bar(metrics, values_lr, color=['#667eea', '#764ba2', '#f093fb', '#f5576c'])
plt.ylim(0, 1.0)
plt.title('Logistic Regression - Performance Metrics', fontsize=14, fontweight='bold')
plt.ylabel('Score', fontsize=12)

for bar, value in zip(bars, values_lr):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
            f'{value:.4f}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

# xgboost model

In [ ]:
xgb_model = xgb.XGBClassifier(
    n_estimators=150,
    max_depth=7,
    learning_rate=0.05,
    random_state=42,
    reg_alpha=1.0, 
    reg_lambda=2.0,
)
xgb_model.fit(
    X_train_tfidf, y_train
)


In [ ]:
# Make predictions
y_pred_xgb = xgb_model.predict(X_test_tfidf)
y_pred_xgb_proba = xgb_model.predict_proba(X_test_tfidf)[:, 1]

# Predict on training data
y_pred_train = xgb_model.predict(X_train_tfidf)

print(f"   Train Accuracy: {accuracy_score(y_train, y_pred_train):.4f}")
print(f"   Test Accuracy:  {accuracy_score(y_test, y_pred_xgb):.4f}")
print(f"   Precision:      {precision_score(y_test, y_pred_xgb):.4f}")
print(f"   Recall:         {recall_score(y_test, y_pred_xgb):.4f}")
print(f"   F1-Score:       {f1_score(y_test, y_pred_xgb):.4f}")
print("\n" + classification_report(y_test, y_pred_xgb, target_names=['Real', 'Fake'], digits=4))

In [ ]:
xgb_accuracy = accuracy_score(y_test, y_pred_xgb)
xgb_precision = precision_score(y_test, y_pred_xgb)
xgb_recall = recall_score(y_test, y_pred_xgb)
xgb_f1 = f1_score(y_test, y_pred_xgb)

# Confusion Matrix
cm_xgb = confusion_matrix(y_test, y_pred_xgb)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Purples', 
            xticklabels=['Real', 'Fake'], 
            yticklabels=['Real', 'Fake'],
            cbar_kws={'label': 'Count'})
plt.title(f'XGBoost - Confusion Matrix\nAccuracy: {xgb_accuracy:.4f}', 
          fontsize=14, fontweight='bold')
plt.ylabel('Actual', fontsize=12)
plt.xlabel('Predicted', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
#bar Chart
values_xgb = [xgb_accuracy, xgb_precision, xgb_recall, xgb_f1]

plt.figure(figsize=(10, 6))
bars = plt.bar(metrics, values_xgb, color=['#667eea', '#764ba2', '#f093fb', '#f5576c'])
plt.ylim(0, 1.0)
plt.title('XGBoost - Performance Metrics', fontsize=14, fontweight='bold')
plt.ylabel('Score', fontsize=12)

for bar, value in zip(bars, values_xgb):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
            f'{value:.4f}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance (Top 20)
importance_xgb = xgb_model.feature_importances_
feature_names = tfidf.get_feature_names_out()
indices = np.argsort(importance_xgb)[-20:]

plt.figure(figsize=(10, 8))
plt.barh(range(len(indices)), importance_xgb[indices], color='#667eea')
plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
plt.xlabel('Feature Importance', fontsize=12)
plt.title('Top 20 Feature Importance - XGBoost', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# MODEL 3: CNN-LSTM

In [ ]:
cnn_lstm_model = Sequential([
Embedding(input_dim=max_words, output_dim=128, input_length=max_len),
    SpatialDropout1D(0.2),

    Conv1D(filters=128, kernel_size=5, activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),
    Dropout(0.3),
    
    Conv1D(filters=64, kernel_size=3, activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),
    Dropout(0.3),

    LSTM(128, dropout=0.2, recurrent_dropout=0.2, return_sequences=True),
    BatchNormalization(),
    
    LSTM(64, dropout=0.2, recurrent_dropout=0.2),
    BatchNormalization(),
    
    # Dense layers with regularization
    Dense(128, activation='relu', kernel_regularizer=l2(0.01)),
    Dropout(0.4),
    
    Dense(64, activation='relu', kernel_regularizer=l2(0.01)),
    Dropout(0.3),
    
    Dense(32, activation='relu', kernel_regularizer=l2(0.01)),
    Dropout(0.2),
    
    # Output
    Dense(1, activation='sigmoid')
])

In [ ]:
# Compile
cnn_lstm_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
callbacks = [
    EarlyStopping(patience=3, restore_best_weights=True),
    ReduceLROnPlateau(factor=0.5, patience=2)
]

In [ ]:
history_cnn_lstm = cnn_lstm_model.fit(
    X_train_pad, y_train,
    validation_split=0.1,
    epochs=5,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
cnn_lstm_loss,cnn_lstm_acc = cnn_lstm_model.evaluate(X_test_pad, y_test, verbose=0)
print(f"cnn-lstm Test Accuracy: {cnn_lstm_acc:.4f}")

In [ ]:
y_pred_cnn_lstm = (cnn_lstm_model.predict(X_test_pad) > 0.5).astype(int)

In [ ]:
print(f"Accuracy:  {accuracy_score(y_test, y_pred_cnn_lstm ):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_cnn_lstm ):.4f}")
print(f"Recall:    {recall_score(y_test,y_pred_cnn_lstm ):.4f}")
print(f"F1-Score:  {f1_score(y_test,y_pred_cnn_lstm ):.4f}")
print("\n" + classification_report(y_test,y_pred_cnn_lstm , target_names=['Real', 'Fake'], digits=4))

In [ ]:
plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
plt.plot(history_cnn_lstm.history['accuracy'], label='Training Accuracy', 
        linewidth=2, marker='o')
plt.plot(history_cnn_lstm.history['val_accuracy'], label='Validation Accuracy', 
        linewidth=2, marker='s')
plt.title('CNN-LSTM - Accuracy', fontsize=14, fontweight='bold')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history_cnn_lstm.history['loss'], label='Training Loss', 
        linewidth=2, marker='o')
plt.plot(history_cnn_lstm.history['val_loss'], label='Validation Loss', 
        linewidth=2, marker='s')
plt.title('CNN-LSTM - Loss', fontsize=14, fontweight='bold')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
cnn_lstm_accuracy = accuracy_score(y_test, y_pred_cnn_lstm)
cnn_lstm_precision = precision_score(y_test, y_pred_cnn_lstm)
cnn_lstm_recall = recall_score(y_test, y_pred_cnn_lstm)
cnn_lstm_f1 = f1_score(y_test, y_pred_cnn_lstm)

# Confusion Matrix
cm_cnn_lstm = confusion_matrix(y_test, y_pred_cnn_lstm)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_cnn_lstm, annot=True, fmt='d', cmap='Purples', 
            xticklabels=['Real', 'Fake'], 
            yticklabels=['Real', 'Fake'],
            cbar_kws={'label': 'Count'})
plt.title(f'CNN-LSTM - Confusion Matrix\nAccuracy: {cnn_lstm_accuracy:.4f}', 
          fontsize=14, fontweight='bold')
plt.ylabel('Actual', fontsize=12)
plt.xlabel('Predicted', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Bar Chart
values_cnn_lstm = [cnn_lstm_accuracy, cnn_lstm_precision, cnn_lstm_recall, cnn_lstm_f1]

plt.figure(figsize=(10, 6))
bars = plt.bar(metrics, values_cnn_lstm, color=['#667eea', '#764ba2', '#f093fb', '#f5576c'])
plt.ylim(0, 1.0)
plt.title('CNN-LSTM - Performance Metrics', fontsize=14, fontweight='bold')
plt.ylabel('Score', fontsize=12)

for bar, value in zip(bars, values_cnn_lstm):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
            f'{value:.4f}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()


# LSTM

In [ ]:
lstm_model = Sequential([
    Embedding(input_dim=max_words, output_dim=128, input_length=max_len),
    SpatialDropout1D(0.2),
    
    LSTM(128, dropout=0.2, recurrent_dropout=0.2, return_sequences=True),
    BatchNormalization(),
    
    LSTM(64, dropout=0.2, recurrent_dropout=0.2),
    BatchNormalization(),
    
    # Dense layers with regularization
    Dense(64, activation='relu', kernel_regularizer=l2(0.01)),
    Dropout(0.4),
    
    Dense(32, activation='relu', kernel_regularizer=l2(0.01)),
    Dropout(0.3),
    
    # Output layer
    Dense(1, activation='sigmoid')
])

In [ ]:
lstm_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
callbacks = [
    EarlyStopping(patience=3, restore_best_weights=True),
    ReduceLROnPlateau(factor=0.5, patience=2)
]


In [ ]:
history_lstm = lstm_model.fit(
    X_train_pad, y_train,
    validation_split=0.1,
    epochs=10,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
lstm_loss, lstm_acc = lstm_model.evaluate(X_test_pad, y_test, verbose=0)
print(f"LSTM Test Accuracy: {lstm_acc:.4f}")

In [ ]:
# Predict
y_pred_lstm = (lstm_model.predict(X_test_pad) > 0.5).astype(int)

In [ ]:
print(f"Accuracy:  {accuracy_score(y_test, y_pred_lstm):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_lstm):.4f}")
print(f"Recall:    {recall_score(y_test,y_pred_lstm):.4f}")
print(f"F1-Score:  {f1_score(y_test,y_pred_lstm):.4f}")
print("\n" + classification_report(y_test, y_pred_lstm, target_names=['Real', 'Fake'], digits=4))

In [ ]:
plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
plt.plot(history_lstm.history['accuracy'], label='Training Accuracy', 
        linewidth=2, marker='o')
plt.plot(history_lstm.history['val_accuracy'], label='Validation Accuracy', 
        linewidth=2, marker='s')
plt.title('LSTM - Accuracy', fontsize=14, fontweight='bold')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history_lstm.history['loss'], label='Training Loss', 
        linewidth=2, marker='o')
plt.plot(history_lstm.history['val_loss'], label='Validation Loss', 
        linewidth=2, marker='s')
plt.title('LSTM - Loss', fontsize=14, fontweight='bold')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
lstm_accuracy = accuracy_score(y_test, y_pred_lstm)
lstm_precision = precision_score(y_test, y_pred_lstm)
lstm_recall = recall_score(y_test, y_pred_lstm)
lstm_f1 = f1_score(y_test, y_pred_lstm)

# Confusion Matrix
cm_lstm = confusion_matrix(y_test, y_pred_lstm)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_lstm, annot=True, fmt='d', cmap='Purples', 
            xticklabels=['Real', 'Fake'], 
            yticklabels=['Real', 'Fake'],
            cbar_kws={'label': 'Count'})
plt.title(f'LSTM - Confusion Matrix\nAccuracy: {lstm_accuracy:.4f}', 
          fontsize=14, fontweight='bold')
plt.ylabel('Actual', fontsize=12)
plt.xlabel('Predicted', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
#Bar Chart
values_lstm = [lstm_accuracy, lstm_precision, lstm_recall, lstm_f1]

plt.figure(figsize=(10, 6))
bars = plt.bar(metrics, values_lstm, color=['#667eea', '#764ba2', '#f093fb', '#f5576c'])
plt.ylim(0, 1.0)
plt.title('LSTM - Performance Metrics', fontsize=14, fontweight='bold')
plt.ylabel('Score', fontsize=12)

for bar, value in zip(bars, values_lstm):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
            f'{value:.4f}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

# BI LSTM

In [ ]:
bi_lstm_model = Sequential([
    
    Embedding(input_dim=max_words, output_dim=128, input_length=max_len),
    SpatialDropout1D(0.2),
    
    Bidirectional(LSTM(64, dropout=0.2, recurrent_dropout=0.2, 
                       return_sequences=True)),
    BatchNormalization(),
    
    Bidirectional(LSTM(32, dropout=0.2, recurrent_dropout=0.2)),
    BatchNormalization(),
    
    # Dense layers with regularization
    Dense(64, activation='relu', kernel_regularizer=l2(0.01)),
    Dropout(0.4),
    
    Dense(32, activation='relu', kernel_regularizer=l2(0.01)),
    Dropout(0.3),
    
    # Output layer
    Dense(1, activation='sigmoid')
])

In [ ]:
bi_lstm_model.compile(
    loss='binary_crossentropy',
    optimizer=Adam(learning_rate=0.001),
    metrics=['accuracy']
)

In [ ]:
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=0.00001
    )
]

In [ ]:
 bi_history = bi_lstm_model.fit(
    X_train_pad, y_train,
    validation_split=0.1,
    epochs=5,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
bi_lstm_loss, bi_lstm_acc = bi_lstm_model.evaluate(X_test_pad, y_test, verbose=0)
print(f"BILSTM Test Accuracy: {bi_lstm_acc:.4f}")

In [ ]:
# Predict
y_pred_bilstm = bi_lstm_model.predict(X_test_pad) > 0.5).astype(int)

In [ ]:
print(f"Accuracy:  {accuracy_score(y_test, y_pred_bilstm):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_bilstm):.4f}")
print(f"Recall:    {recall_score(y_test,y_pred_bilstm):.4f}")
print(f"F1-Score:  {f1_score(y_test,y_pred_bilstm):.4f}")
print("\n" + classification_report(y_test, y_pred_bilstm, target_names=['Real', 'Fake'], digits=4))

In [ ]:
plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
plt.plot(bi_history.history['accuracy'], label='Training Accuracy', 
        linewidth=2, marker='o')
plt.plot(bi_history.history['val_accuracy'], label='Validation Accuracy', 
        linewidth=2, marker='s')
plt.title('BiLSTM - Accuracy', fontsize=14, fontweight='bold')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(bi_history.history['loss'], label='Training Loss', 
        linewidth=2, marker='o')
plt.plot(bi_history.history['val_loss'], label='Validation Loss', 
        linewidth=2, marker='s')
plt.title('BiLSTM - Loss', fontsize=14, fontweight='bold')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
bilstm_accuracy = accuracy_score(y_test, y_pred_bilstm)
bilstm_precision = precision_score(y_test, y_pred_bilstm)
bilstm_recall = recall_score(y_test, y_pred_bilstm)
bilstm_f1 = f1_score(y_test, y_pred_bilstm)

# Confusion Matrix
cm_bilstm = confusion_matrix(y_test, y_pred_bilstm)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_bilstm, annot=True, fmt='d', cmap='Purples', 
            xticklabels=['Real', 'Fake'], 
            yticklabels=['Real', 'Fake'],
            cbar_kws={'label': 'Count'})
plt.title(f'BiLSTM - Confusion Matrix\nAccuracy: {bilstm_accuracy:.4f}', 
          fontsize=14, fontweight='bold')
plt.ylabel('Actual', fontsize=12)
plt.xlabel('Predicted', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
values_bilstm = [bilstm_accuracy, bilstm_precision, bilstm_recall, bilstm_f1]

plt.figure(figsize=(10, 6))
bars = plt.bar(metrics, values_bilstm, color=['#667eea', '#764ba2', '#f093fb', '#f5576c'])
plt.ylim(0, 1.0)
plt.title('BiLSTM - Performance Metrics', fontsize=14, fontweight='bold')
plt.ylabel('Score', fontsize=12)

for bar, value in zip(bars, values_bilstm):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
            f'{value:.4f}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

# Bert

In [ ]:
bert_model = TFDistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2  
)

optimizer = AdamWeightDecay(
    learning_rate=2e-5,
    weight_decay_rate=0.01,
    epsilon=1e-08,
    exclude_from_weight_decay=["LayerNorm", "bias"]
)

In [ ]:
bert_model.compile(
    optimizer=optimizer,
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

In [ ]:
 history_bert = bert_model.fit(
    X_train_encoded, y_train,
    validation_split=0.1,
    epochs=5,
    batch_size=32,
    verbose=1
)

In [ ]:
bert_loss, bert_acc = bert_model.evaluate(X_test_encoded, y_test, verbose=0)
print(f"BERT Test Accuracy: {bert_acc:.4f}")

In [ ]:
predictions = bert_model.predict(X_test_encoded)
y_pred_bert = np.argmax(predictions.logits, axis=1)

# Get probabilities for ROC/AUC if needed
probs = tf.nn.softmax(predictions.logits, axis=-1).numpy()
y_pred_probs = probs[:, 1]

In [ ]:
print(f"Accuracy:  {accuracy_score(y_test, y_pred_bert):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_bert):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_bert):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_bert):.4f}")
print("\n" + classification_report(y_test, y_pred_bert, target_names=['Real', 'Fake'], digits=4))

In [ ]:
plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
plt.plot(history_bert.history['accuracy'], label='Training Accuracy', 
        linewidth=2, marker='o')
plt.plot(history_bert.history['val_accuracy'], label='Validation Accuracy', 
        linewidth=2, marker='s')
plt.title('BERT - Accuracy', fontsize=14, fontweight='bold')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history_bert.history['loss'], label='Training Loss', 
        linewidth=2, marker='o')
plt.plot(history_bert.history['val_loss'], label='Validation Loss', 
        linewidth=2, marker='s')
plt.title('BERT - Loss', fontsize=14, fontweight='bold')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
bert_accuracy = accuracy_score(y_test, y_pred_bert)
bert_precision = precision_score(y_test, y_pred_bert)
bert_recall = recall_score(y_test, y_pred_bert)
bert_f1 = f1_score(y_test, y_pred_bert)

# Confusion Matrix
cm_bert = confusion_matrix(y_test, y_pred_bert)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_bert, annot=True, fmt='d', cmap='Purples', 
            xticklabels=['Real', 'Fake'], 
            yticklabels=['Real', 'Fake'],
            cbar_kws={'label': 'Count'})
plt.title(f'BERT - Confusion Matrix\nAccuracy: {bert_accuracy:.4f}', 
          fontsize=14, fontweight='bold')
plt.ylabel('Actual', fontsize=12)
plt.xlabel('Predicted', fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
values_bert = [bert_accuracy, bert_precision, bert_recall, bert_f1]

plt.figure(figsize=(8, 10))
bars = plt.bar(metrics, values_bert, color=['#667eea', '#764ba2', '#f093fb', '#f5576c'])
plt.ylim(0, 1.0)
plt.title('BERT - Performance Metrics', fontsize=14, fontweight='bold')
plt.ylabel('Score', fontsize=12)

for bar, value in zip(bars, values_bert):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
            f'{value:.4f}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
bert_model.save_pretrained('bert_model')

# model comparison

In [ ]:
results = {
    'Model': ['Logistic Regression', 'XGBoost', 'LSTM', 'CNN-LSTM', 'BERT'],
    'Accuracy': [lr_accuracy, xgb_accuracy, lstm_accuracy, 
                 cnn_lstm_accuracy, bert_accuracy],
    'Precision': [lr_precision, xgb_precision, lstm_precision, 
                  cnn_lstm_precision, bert_precision],
    'Recall': [lr_recall, xgb_recall, lstm_recall,
               cnn_lstm_recall, bert_recall],
    'F1-Score': [lr_f1, xgb_f1, lstm_f1, cnn_lstm_f1, bert_f1]
}

results_df = pd.DataFrame(results)

print(results_df.to_string(index=False))

In [ ]:
#Plot comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
comparison_metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
colors = ['#667eea', '#764ba2', '#f093fb', '#f5576c']

for idx, (ax, metric, color) in enumerate(zip(axes.flat, comparison_metrics, colors)):
    values = results_df[metric].values
    bars = ax.barh(results_df['Model'], values, color=color)
    ax.set_xlabel('Score', fontsize=11)
    ax.set_title(f'{metric} Comparison', fontsize=13, fontweight='bold')
    ax.set_xlim(0.90, 1.0)
    
    # Add value labels
    for bar, value in zip(bars, values):
        width = bar.get_width()
        ax.text(width, bar.get_y() + bar.get_height()/2.,
                f'{value:.4f}',
                ha='left', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Best model
best_model_idx = results_df['Accuracy'].idxmax()
best_model = results_df.iloc[best_model_idx]

print(f" BEST MODEL: {best_model['Model']}")
print(f"   Accuracy:  {best_model['Accuracy']:.4f}")
print(f"   Precision: {best_model['Precision']:.4f}")
print(f"   Recall:    {best_model['Recall']:.4f}")
print(f"   F1-Score:  {best_model['F1-Score']:.4f}")

In [ ]:
y_pred_lr_proba = lr_model.predict_proba(X_test_tfidf)[:, 1]
y_pred_xgb_proba = xgb_model.predict_proba(X_test_tfidf)[:, 1]
y_pred_lstm_proba = lstm_model.predict(X_test_pad).flatten()
y_pred_cnn_lstm_proba = cnn_lstm_model.predict(X_test_pad).flatten()

In [ ]:
# ROC curves
plt.figure(figsize=(10, 8))

colors_roc = ['#667eea', '#764ba2', '#f093fb',  '#4facfe', '#00f2fe']
model_names_roc = ['Logistic Regression', 'XGBoost', 'LSTM', 'CNN-LSTM', 'BERT']
model_probs = [y_pred_lr_proba, y_pred_xgb_proba, y_pred_lstm_proba, 
               y_pred_cnn_lstm_proba, y_pred_probs]

for name, probs, color in zip(model_names_roc, model_probs, colors_roc):
    fpr, tpr, _ = roc_curve(y_test, probs)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, linewidth=2, label=f'{name} (AUC = {roc_auc:.4f})')

plt.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves - Model Comparison', fontsize=14, fontweight='bold')
plt.legend(loc="lower right", fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Word cloud for Real news
real_text = ' '.join(df[df['label'] == 0]['clean_text'].values)
wc_real = WordCloud(width=800, height=400, background_color='white',
                   colormap='Blues', max_words=100).generate(real_text)

# Word cloud for Fake news
fake_text = ' '.join(df[df['label'] == 1]['clean_text'].values)
wc_fake = WordCloud(width=800, height=400, background_color='white',
                   colormap='Reds', max_words=100).generate(fake_text)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].imshow(wc_real, interpolation='bilinear')
axes[0].set_title('Real News - Word Cloud', fontsize=14, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(wc_fake, interpolation='bilinear')
axes[1].set_title('Fake News - Word Cloud', fontsize=14, fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 5))

epochs = range(1, len(history_bert.history['accuracy']) + 1)

plt.subplot(1, 2, 1)
plt.plot(epochs, history_bert.history['accuracy'], 'b-o', label='Training', linewidth=2)
plt.plot(epochs, history_bert.history['val_accuracy'], 'r-s', label='Validation', linewidth=2)
plt.fill_between(epochs, history_bert.history['accuracy'], history_bert.history['val_accuracy'], 
                 alpha=0.2, color='gray')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('BERT Learning Curve - Accuracy', fontsize=13, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(epochs, history_bert.history['loss'], 'b-o', label='Training', linewidth=2)
plt.plot(epochs, history_bert.history['val_loss'], 'r-s', label='Validation', linewidth=2)
plt.fill_between(epochs, history_bert.history['loss'], history_bert.history['val_loss'], 
                 alpha=0.2, color='gray')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('BERT Learning Curve - Loss', fontsize=13, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Calculate overfitting
train_val_gap = history_bert.history['accuracy'][-1] - history_bert.history['val_accuracy'][-1]
print(f"\nOverfitting Analysis:")
print(f"Final Training Accuracy: {history_bert.history['accuracy'][-1]:.4f}")
print(f"Final Validation Accuracy: {history_bert.history['val_accuracy'][-1]:.4f}")
print(f"Train-Val Gap: {train_val_gap:.4f}")
if train_val_gap < 0.02:
    print(" Model is well-generalized (low overfitting)")
elif train_val_gap < 0.05:
    print(" Model shows slight overfitting")
else:
    print(" Model is overfitting significantly")

In [ ]:
#PRECISION-RECALL CURVE
plt.figure(figsize=(10, 8))

colors_pr = ['#667eea', '#764ba2', '#f093fb', '#f5576c', '#4facfe', '#00f2fe']
model_names_pr = ['Logistic Regression', 'XGBoost', 'LSTM', 'BiLSTM', 'CNN-LSTM', 'BERT']
model_probs_pr = [y_pred_lr_proba, y_pred_xgb_proba, y_pred_lstm_proba, 
                   y_pred_cnn_lstm_proba, y_pred_probs]

for name, probs, color in zip(model_names_pr, model_probs_pr, colors_pr):
    precision, recall, _ = precision_recall_curve(y_test, probs)
    avg_precision = average_precision_score(y_test, probs)
    plt.plot(recall, precision, color=color, linewidth=2, 
            label=f'{name} (AP = {avg_precision:.4f})')

plt.xlabel('Recall', fontsize=12)
plt.ylabel('Precision', fontsize=12)
plt.title('Precision-Recall Curves - Model Comparison', fontsize=14, fontweight='bold')
plt.legend(loc="lower left", fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()